## 2.2 데이터 준비와 모델 구성

In [2]:
"""
데이터셋 로드

목적:
- 네이버 뉴스 요약 데이터셋을 Hugging Face Hub에서 불러옵니다.

동작:
- load_dataset(...) 호출 시 train/validation/test split을 포함한 DatasetDict를 구성합니다.
- 최초 실행 시 로컬 캐시에 다운로드가 일어날 수 있습니다.
"""
from datasets import load_dataset

dataset = load_dataset("daekeun-ml/naver-news-summarization-ko")

In [3]:
"""
데이터셋 참조 변수 확인

목적:
- dataset 객체를 data라는 이름으로 참조해 이후 셀에서 재사용하기 쉽게 만듭니다.

동작:
- data = dataset 은 복사가 아니라 같은 객체를 가리키는 참조 대입입니다.
- 마지막 줄 data 를 통해 노트북 출력으로 DatasetDict 구조를 확인합니다.
"""
data = dataset
data

DatasetDict({
    train: Dataset({
        features: ['date', 'category', 'press', 'title', 'document', 'link', 'summary'],
        num_rows: 22194
    })
    validation: Dataset({
        features: ['date', 'category', 'press', 'title', 'document', 'link', 'summary'],
        num_rows: 2466
    })
    test: Dataset({
        features: ['date', 'category', 'press', 'title', 'document', 'link', 'summary'],
        num_rows: 2740
    })
})

불러온 데이터셋은 DatasetDict로 구성되어 있고, train, validation, test의 세 가지 세트로 나뉘어 있음

각 세트는 동일한 특성을 가지고 있으며, 이는

- **날짜 date**
- **카테고리 category**
- **언론사 press**
- **제목 title**
- **기사 본문 document**
- **링크 link**
- **요약 summary**

으로 구성되어 있음

In [4]:
"""
첫 번째 기사 본문 샘플 확인

목적:
- train split의 첫 샘플에서 document 텍스트를 직접 확인합니다.

동작:
- data["train"]["document"][0] 인덱싱으로 첫 문서를 문자열로 조회합니다.
"""
data["train"]["document"][0]

'앵커 정부가 올해 하반기 우리 경제의 버팀목인 수출 확대를 위해 총력을 기울이기로 했습니다. 특히 수출 중소기업의 물류난 해소를 위해 무역금융 규모를 40조 원 이상 확대하고 물류비 지원과 임시선박 투입 등을 추진하기로 했습니다. 류환홍 기자가 보도합니다. 기자 수출은 최고의 실적을 보였지만 수입액이 급증하면서 올해 상반기 우리나라 무역수지는 역대 최악인 103억 달러 적자를 기록했습니다. 정부가 수출확대에 총력을 기울이기로 한 것은 원자재 가격 상승 등 대외 리스크가 가중되는 상황에서 수출 증가세 지속이야말로 한국경제의 회복을 위한 열쇠라고 본 것입니다. 추경호 경제부총리 겸 기획재정부 장관 정부는 우리 경제의 성장엔진인 수출이 높은 증가세를 지속할 수 있도록 총력을 다하겠습니다. 우선 물류 부담 증가 원자재 가격 상승 등 가중되고 있는 대외 리스크에 대해 적극 대응하겠습니다. 특히 중소기업과 중견기업 수출 지원을 위해 무역금융 규모를 연초 목표보다 40조 원 늘린 301조 원까지 확대하고 물류비 부담을 줄이기 위한 대책도 마련했습니다. 이창양 산업통상자원부 장관 국제 해상운임이 안정될 때까지 월 4척 이상의 임시선박을 지속 투입하는 한편 중소기업 전용 선복 적재 용량 도 현재보다 주당 50TEU 늘려 공급하겠습니다. 하반기에 우리 기업들의 수출 기회를 늘리기 위해 2 500여 개 수출기업을 대상으로 해외 전시회 참가를 지원하는 등 마케팅 지원도 벌이기로 했습니다. 정부는 또 이달 중으로 반도체를 비롯한 첨단 산업 육성 전략을 마련해 수출 증가세를 뒷받침하고 에너지 소비를 줄이기 위한 효율화 방안을 마련해 무역수지 개선에 나서기로 했습니다. YTN 류환홍입니다.'

In [5]:
"""
샘플 문서의 고유 문자 집합 확인

목적:
- 첫 번째 기사 본문에 어떤 문자들이 등장하는지 확인합니다.

동작:
- set(...)으로 중복 문자를 제거합니다.
- sorted(...)로 정렬해 사람이 보기 쉬운 순서로 출력합니다.
"""
print(sorted(list(set(data["train"]["document"][0]))))

[' ', '.', '0', '1', '2', '3', '4', '5', 'E', 'N', 'T', 'U', 'Y', '가', '개', '것', '겠', '격', '견', '겸', '경', '고', '공', '과', '관', '국', '규', '극', '금', '급', '기', '까', '나', '난', '너', '높', '는', '늘', '니', '다', '단', '달', '담', '당', '대', '도', '되', '될', '뒷', '들', '등', '때', '또', '라', '략', '량', '러', '려', '력', '련', '로', '록', '롯', '류', '를', '리', '린', '마', '만', '말', '면', '모', '목', '무', '물', '박', '반', '받', '방', '버', '벌', '보', '복', '본', '부', '비', '산', '상', '서', '선', '성', '세', '소', '속', '쇠', '수', '스', '습', '승', '시', '실', '악', '안', '액', '앵', '야', '양', '억', '업', '에', '엔', '여', '역', '연', '열', '였', '올', '외', '용', '우', '운', '울', '원', '월', '위', '육', '율', '융', '으', '은', '을', '응', '의', '이', '인', '임', '입', '있', '자', '장', '재', '적', '전', '정', '제', '조', '주', '줄', '중', '증', '지', '진', '참', '창', '책', '척', '첨', '체', '초', '총', '최', '추', '출', '침', '커', '케', '크', '통', '투', '특', '팀', '팅', '편', '표', '하', '한', '할', '합', '해', '했', '현', '호', '홍', '화', '확', '환', '황', '회', '획', '효', '히']


In [6]:
"""
전체 학습 문서 기준 문자 사전 크기 계산

목적:
- train 문서 전체에서 문자 단위 vocabulary를 구성하고 크기를 계산합니다.

동작:
- 모든 문서를 하나의 긴 문자열로 합칩니다.
- set(...), sorted(...)로 고유 문자 목록을 만들고 len(...)으로 개수를 계산합니다.
"""

ko_text = "".join(data["train"]["document"])
ko_chars = sorted(list(set((ko_text))))
ko_vocab_size = len(ko_chars)
print("총 글자 수 :", ko_vocab_size)

총 글자 수 : 2701


In [7]:
"""
문자 사전 일부 구간 확인

목적:
- 문자 사전의 특정 인덱스 구간을 확인해 정렬 결과를 점검합니다.

동작:
- ko_chars[2000:2100] 슬라이싱으로 중간 구간 문자들을 출력합니다.
"""

print(ko_chars[2000:2100])

['왓', '왔', '왕', '왜', '왠', '외', '왹', '왼', '요', '욕', '욘', '욜', '욤', '욥', '용', '우', '욱', '운', '욷', '울', '움', '웁', '웃', '웅', '워', '웍', '원', '월', '웜', '웠', '웡', '웨', '웬', '웰', '웸', '웹', '웻', '위', '윅', '윈', '윌', '윔', '윕', '윗', '윙', '유', '육', '윤', '율', '윱', '윳', '융', '으', '윽', '은', '을', '음', '읍', '읏', '응', '의', '읠', '이', '익', '인', '일', '읽', '잃', '임', '입', '잇', '있', '잉', '잊', '잎', '자', '작', '잔', '잖', '잘', '잠', '잡', '잣', '잤', '장', '잦', '재', '잭', '잰', '잼', '잽', '잿', '쟁', '쟈', '쟝', '쟤', '저', '적', '전', '절']


In [8]:
"""
문자-정수 매핑 함수 구성

목적:
- 문자 단위 토크나이징을 위한 인코딩/디코딩 매핑을 정의합니다.

동작:
- 문자->id, id->문자 사전을 각각 만듭니다.
- token_encode/token_decode 람다를 정의하고 샘플 문장으로 왕복 검증합니다.
"""
character_to_ids = {char: i for i, char in enumerate(ko_chars)}
ids_to_character = {i: char for i, char in enumerate(ko_chars)}
token_encode = lambda s: [character_to_ids[c] for c in s]
token_decode = lambda l: "".join([ids_to_character[i] for i in l])
print(token_encode("안녕하세요 함께 인공지능을 공부하게 되어 반가워요."))
print(token_decode(token_encode("안녕하세요 함께 인공지능을 공부하게 되어 반가워요.")))

[1909, 1169, 2546, 1770, 2008, 0, 2551, 1061, 0, 2064, 977, 2157, 1209, 2055, 0, 977, 1658, 2546, 949, 0, 1283, 1942, 0, 1593, 908, 2024, 2008, 2]
안녕하세요 함께 인공지능을 공부하게 되어 반가워요.


In [9]:
""" (장표)
전체 텍스트 정수 토큰화

목적:
- 전체 학습 텍스트를 모델 입력용 정수 텐서로 변환합니다.

동작:
- token_encode(ko_text) 결과를 torch.long 타입 텐서로 생성합니다.
- shape/dtype과 앞부분 토큰 값을 확인합니다.
"""

import torch

tokenized_data = torch.tensor(token_encode(ko_text), dtype=torch.long)
print(tokenized_data.shape, tokenized_data.dtype)
print(tokenized_data[:100])

torch.Size([22162967]) torch.int64
tensor([1928, 2315,    0, 2105, 1658,  908,    0, 1987, 2555,    0, 2546, 1593,
        1028,    0, 2015, 1485,    0,  965, 2107, 2060,    0, 1617, 2465, 1542,
        2064,    0, 1808, 2273,    0, 2603, 1236, 1477,    0, 2037, 2555,    0,
        2263, 1430, 2055,    0, 1028, 2019, 2062, 1028, 1441,    0, 2562, 1841,
        1213, 1221,    2,    0, 2451, 2650,    0, 1808, 2273,    0, 2142, 1787,
        1028, 1950, 2060,    0, 1558, 1468, 1119,    0, 2555, 1787, 1477,    0,
        2037, 2555,    0, 1553, 1967, 1024, 2051,    0, 1015, 1541, 1477,    0,
           7,    3, 2117,    0, 2026,    0, 2062, 1740,    0, 2603, 1236, 2546,
         968,    0, 1558, 1468])


In [10]:
"""
학습/평가 구간 분할

목적:
- 토큰 시퀀스를 비율 기준으로 train/test 구간으로 분리합니다.

동작:
- 전체 길이의 90% 지점을 경계로 앞부분은 train, 뒷부분은 test로 나눕니다.
"""

n = int(0.9 * len(tokenized_data))
train_dataset = tokenized_data[:n]
test_dataset = tokenized_data[n:]

In [11]:
"""
블록 크기 샘플 확인

목적:
- 컨텍스트 길이(block_size) 개념을 간단한 예시로 확인합니다.

동작:
- block_size를 8로 설정하고 학습 데이터의 첫 8개 토큰을 조회합니다.
"""

block_size = 8
train_dataset[:block_size]

tensor([1928, 2315,    0, 2105, 1658,  908,    0, 1987])

In [12]:
"""
다음 토큰 예측용 x/y 정렬 확인

목적:
- 언어모델 학습에서 입력(x)과 정답(y)이 한 칸 시프트된 관계임을 확인합니다.

동작:
- x는 현재 위치까지의 컨텍스트, y는 다음 위치 토큰으로 구성합니다.
- 시간축을 순회하며 각 시점의 입력 컨텍스트와 타깃 토큰을 출력합니다.
"""
x = train_dataset[:block_size]
y = train_dataset[1:block_size + 1]

for time in range(block_size):
    context = x[:time + 1]
    target = y[time]

    print(f"입력 텐셔 : {context}")
    print(f"타켓 글자 : {target}")

입력 텐셔 : tensor([1928])
타켓 글자 : 2315
입력 텐셔 : tensor([1928, 2315])
타켓 글자 : 0
입력 텐셔 : tensor([1928, 2315,    0])
타켓 글자 : 2105
입력 텐셔 : tensor([1928, 2315,    0, 2105])
타켓 글자 : 1658
입력 텐셔 : tensor([1928, 2315,    0, 2105, 1658])
타켓 글자 : 908
입력 텐셔 : tensor([1928, 2315,    0, 2105, 1658,  908])
타켓 글자 : 0
입력 텐셔 : tensor([1928, 2315,    0, 2105, 1658,  908,    0])
타켓 글자 : 1987
입력 텐셔 : tensor([1928, 2315,    0, 2105, 1658,  908,    0, 1987])
타켓 글자 : 2555


In [ ]:
"""
미니배치 생성 함수 정의

목적:
- 학습/평가용 랜덤 미니배치를 만드는 배치 함수를 정의합니다.

동작:
- 임의 시작 인덱스를 뽑아 길이 block_size의 x/y 시퀀스를 여러 개 쌓습니다.
- batch_size x block_size 형태를 확인하고 샘플 배치 내용을 출력합니다.
"""
torch.manual_seed(1234)

batch_size = 4
block_size = 8


def batch_function(mode):
    dataset = train_dataset if mode == "train" else test_dataset

    idx = torch.randint(len(dataset) - block_size, (batch_size,))
    """
    - 배치 크기만큼 시작 인덱스 뽑음
    - 각 시작점이 block_size 길이 슬라이스를 안전하게 만들 수 있게 범위 줄임
    """

    x = torch.stack([dataset[index:index + block_size] for index in idx])
    """
    - 각 시작 인덱스에서 index:index+block_size 구간 자름
    - 잘린 시퀀스들을 쌓아서 (batch_size, block_size) 텐서 만듦
    """

    y = torch.stack([dataset[index + 1:index + block_size + 1] for index in idx])
    """
    - 각 시작 인덱스에서 index+1:index+block_size+1 구간 자름
    - x보다 한 칸 오른쪽으로 민 정답 시퀀스 만듦
    - shape 역시 (batch_size, block_size) 만듦
    """

    return x, y


"""
- 코드가 실제로 만드는 대상은 x 텐서임
- x.shape를 출력할 때 파이토치가 모양을 (batch_size, block_size) 튜플로 보여줌
- 즉 튜플은 설명용 표기이고 학습 데이터 본체는 2차원 텐서임

흐름을 아주 작게 풀어보면 이해 쉬움
batch_size = 3
block_size = 4
idx = [10, 20, 30]  # 시작 위치 3개 뽑음

각 인덱스마다 슬라이스 만듦
- dataset[10:14]  -> 길이 4 벡터
- dataset[20:24]  -> 길이 4 벡터
- dataset[30:34]  -> 길이 4 벡터

이 시점 리스트 형태는 이런 느낌
[
  tensor([a, b, c, d]),   # 길이 4
  tensor([e, f, g, h]),   # 길이 4
  tensor([i, j, k, l])    # 길이 4
]

torch.stack(...)이 하는 일
- 위 1차원 텐서 3개를 새로운 축으로 쌓음
- 결과를 2차원 텐서로 만듦
- 행 개수는 3  -> batch_size
- 열 개수는 4  -> block_size

그래서
- 실제 데이터 구조: 2차원 텐서
- 그 텐서의 shape 표기: (3, 4)
- 일반화하면 (batch_size, block_size)가 됨

한 줄 요약
- batch_size는 몇 개 시퀀스를 한 번에 담는지 의미
- block_size는 시퀀스 하나의 길이를 의미
- stack이 이 둘을 축으로 갖는 2차원 텐서 만듦
"""

In [14]:
example_x, example_y = batch_function("train")
print("inputs : ", example_x.shape)
print("")
print("example_x의 실제 값")
print(example_x)
print("-----------------------")
print("targets : ", example_y.shape)
print("")
print("example_y의 실제 값")
print(example_y)
print("-----------------------")

for size in range(batch_size):
    for t in range(block_size):
        context = example_x[size, :t + 1]  #  context = x[:t+1]로 슬라이싱해서 길이를 1개씩 늘림
        target = example_y[size, t]
        print(f"input : {context}, target : {target}")
    print("-----------------------")
    print("-----------------------")

inputs :  torch.Size([4, 8])

example_x의 실제 값
tensor([[1764, 2555,    0, 1236, 2248,    0, 2017, 1976],
        [   0, 1966, 2157,    0, 1951, 2062,    0, 2548],
        [   0, 1304, 1485, 1586,    0, 1907, 2450,    0],
        [   3,    2,    6,    5,    1,    0,    5,    3]])
-----------------------
targets :  torch.Size([4, 8])

example_y의 실제 값
tensor([[2555,    0, 1236, 2248,    0, 2017, 1976, 2546],
        [1966, 2157,    0, 1951, 2062,    0, 2548, 2289],
        [1304, 1485, 1586,    0, 1907, 2450,    0, 2480],
        [   2,    6,    5,    1,    0,    5,    3,    5]])
-----------------------
input : tensor([1764]), target : 2555
input : tensor([1764, 2555]), target : 0
input : tensor([1764, 2555,    0]), target : 1236
input : tensor([1764, 2555,    0, 1236]), target : 2248
input : tensor([1764, 2555,    0, 1236, 2248]), target : 0
input : tensor([1764, 2555,    0, 1236, 2248,    0]), target : 2017
input : tensor([1764, 2555,    0, 1236, 2248,    0, 2017]), target : 1976
input :

### 저자가 보여주고자 한 핵심은 다음 3가지

- 언어모델 학습 목표를 직관화
  - 지금까지 본 토큰들로 다음 토큰 하나 맞추는 구조 보여줌
- x와 y의 한 칸 시프트 관계를 체감
  - 입력 시퀀스와 정답 시퀀스가 어떻게 짝지어지는지 눈으로 확인시킴
- 한 시퀀스 안에서도 학습 신호가 여러 개 생기는지,
  - 길이 8 블록이면 예측 과제 8개 만듦
  - 1764 -> 2555, 1764,2555 -> 0 같은 형태로 과제 누적시킴

즉 모델이 문장을 통째로 한 번에 외우는 게 아니라, **접두사 컨텍스트를 점점 늘려가며 다음 토큰 예측 문제를 반복 학습한다는 점 강조**

In [15]:
"""
핵심 텐서 요약 확인

목적:
- 현재 실험의 주요 값(어휘 크기, 배치 shape, 타깃 텐서)을 한 번에 점검

동작:
- ko_vocab_size, example_x.shape, example_y를 튜플로 출력
"""
ko_vocab_size, example_x.shape, example_y

(2701,
 torch.Size([4, 8]),
 tensor([[2555,    0, 1236, 2248,    0, 2017, 1976, 2546],
         [1966, 2157,    0, 1951, 2062,    0, 2548, 2289],
         [1304, 1485, 1586,    0, 1907, 2450,    0, 2480],
         [   2,    6,    5,    1,    0,    5,    3,    5]]))

# 2.3 언어모델 만들기

## 2.3.1 ~ 2.3.2 라이브러리 설명 & _ _ init _ _ 함수 

In [16]:
"""
초기 semiGPT 모델 정의 (장표)

목적:
- 가장 단순한 문자 기반 언어모델 골격을 정의합니다.

동작:
- Embedding 테이블 하나로 입력 토큰을 바로 vocabulary 차원 logits로 변환
- forward 결과 shape를 확인해 배치/시퀀스/어휘 축을 점검
"""
import torch
import torch.nn as nn
from torch.nn import functional as F


class semiGPT(nn.Module):
    def __init__(self, vocab_length):
        super().__init__()
        self.embedding_token_table = nn.Embedding(vocab_length, vocab_length)

    def forward(self, inputs, targets):
        logits = self.embedding_token_table(inputs)

        return logits


model = semiGPT(ko_vocab_size)
output = model(example_x, example_y)
print(output.shape)

torch.Size([4, 8, 2701])


## 2.3.3 forward 메서드

모델에 데이터가 전달될 때 호출되어 실제 계산 과정을 담당

In [17]:
"""
forward에서 손실 계산 추가

목적:
- 모델 forward 내부에서 loss를 함께 계산하도록 확장

동작:
- logits를 (batch*seq, vocab)으로 평탄화하고 targets도 (batch*seq)로 맞춤
- F.cross_entropy(...)로 손실을 계산하고 shape를 출력해 검증
"""
import torch
import torch.nn as nn
from torch.nn import functional as F


class semiGPT(nn.Module):
    def __init__(self, vocab_length):
        super().__init__()
        self.embedding_token_table = nn.Embedding(vocab_length, vocab_length)

    def forward(self, inputs, targets):
        logits = self.embedding_token_table(inputs)
        batch, seq_length, vocab_length = logits.shape
        # logits는 예측값을 나타냄
        logits = logits.view(batch * seq_length, vocab_length)
        # targets는 실제 값을 나타냄
        targets = targets.view(batch * seq_length)
        loss = F.cross_entropy(logits, targets)

        print("logits의 shape는 : ", logits.shape, "입니다.")
        print("targets의 shape는 : ", targets.shape, "입니다.")

        return logits, loss


model = semiGPT(ko_vocab_size)
logits, loss = model(example_x, example_y)
print(loss)

logits의 shape는 :  torch.Size([32, 2701]) 입니다.
targets의 shape는 :  torch.Size([32]) 입니다.
tensor(8.5332, grad_fn=<NllLossBackward0>)


### loss를 사용하는 이유 (장표)

_"학습을 위해서 사용한다."_


loss는 모델이 “얼마나 틀렸는지”를 수치화한 값이고, 이 값으로 역전파(backward)를 해서 가중치를 업데이트

모델 forward에서 loss까지 계산하면 좋은 이유는:

1. 학습 루프가 단순해짐

- logits 받고 바깥에서 다시 loss 계산하는 코드가 줄어듬


2. 정답 처리 규칙을 모델 내부에 일관되게 캡슐화
- 예: 언어모델의 label shift, padding 무시(ignore_index) 같은 실수하기 쉬운 로직을 한곳에서 처리

## 2.3.4 generate 메서드

In [18]:
"""
generate 메서드 추가

목적:
- 모델이 학습한 패턴을 바탕으로 새로운 텍스트를 생성
- 학습된 모델로 다음 토큰을 순차 생성하는 샘플링 루프를 구현


동작:
- forward를 targets 없이 호출해 logits만 받고, 마지막 시점 분포를 사용
- softmax -> multinomial 샘플링 -> 입력 뒤에 이어 붙이기를 반복
"""
import torch
import torch.nn as nn
from torch.nn import functional as F


class semiGPT(nn.Module):
    def __init__(self, vocab_length):
        super().__init__()
        self.embedding_token_table = nn.Embedding(vocab_length, vocab_length)

    def forward(self, inputs, targets=None):
        logits = self.embedding_token_table(inputs)
        if targets is None:
            loss = None
        else:
            batch, seq_length, vocab_length = logits.shape
            logits = logits.view(batch * seq_length, vocab_length)
            targets = targets.view(batch * seq_length)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, inputs, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, loss = self.forward(inputs)
            logits = logits[:, -1, :]
            print(logits.shape)
            probs = F.softmax(logits, dim=-1)
            next_inputs = torch.multinomial(probs, num_samples=1)
            inputs = torch.cat((inputs, next_inputs), dim=1)
        return inputs


model = semiGPT(ko_vocab_size)
logits, loss = model(example_x, example_y)
print(loss)

token_decode(model.generate(torch.zeros((1, 1),
                                        dtype=torch.long),
                            max_new_tokens=10)[0].tolist())

tensor(8.5369, grad_fn=<NllLossBackward0>)
torch.Size([1, 2701])
torch.Size([1, 2701])
torch.Size([1, 2701])
torch.Size([1, 2701])
torch.Size([1, 2701])
torch.Size([1, 2701])
torch.Size([1, 2701])
torch.Size([1, 2701])
torch.Size([1, 2701])
torch.Size([1, 2701])


' 오춘令勝릐쌔탁숲동c'

In [19]:
"""
마지막 시점 선택 슬라이싱 예시

- logits[:, -1, :] 슬라이싱이 무엇을 선택하는지 작은 텐서 예제로 설명

- 3개 시점 중 마지막 시점 벡터만 추출해 shape 변화를 확인
"""
import torch

logits = torch.tensor(
    [
        [
            [0.1, 0.2, 0.3, 0.4],
            [0.2, 0.3, 0.4, 0.1],
            [0.3, 0.4, 0.1, 0.2]
        ]
    ]
)

result = logits[:, -1, :]
print("선택되는 값        : ", result)
print("결과에 대한 size 값 : ", result.size())

선택되는 값        :  tensor([[0.3000, 0.4000, 0.1000, 0.2000]])
결과에 대한 size 값 :  torch.Size([1, 4])


이 텐서의 차원은 (1, 3, 4)

- 1: 배치 크기
- 3: 시퀀스 길이
- 4: 각 토큰에 대한 로짓 값의 수

logits[:, -1, :] 연산을 수행할 때,

- : (첫 번째 차원): 모든 배치를 선택합니다(여기서는 1개).
- -1 (두 번째 차원): 각 배치의 마지막 시퀀스/토큰을 선택합니다.
- : (세 번째 차원): 선택된 토큰의 모든 로짓 값을 가져옵니다.

이 연산으로 인해 두 번째 차원(시퀀스 차원)이 제거됨

왜냐하면 -1로 인해 이 차원에서 단 하나의 요소만 선택되기 때문

결과적으로,

- 첫 번째 차원은 그대로 유지됩니다(여전히 1개의 배치).
- 두 번째 차원은 사라집니다(단일 시퀀스/토큰 선택으로 인해).
- 세 번째 차원은 그대로 유지됩니다(4개의 로짓 값).

따라서 결과 텐서의 형태는 (1, 4)가 되며, 이는 2차원 텐서

## 2.4 optimizer 추가하기

In [20]:
"""
옵티마이저 초기화

목적:
- 모델 학습을 위한 AdamW 옵티마이저를 설정

Adam은 적응형 학습률을 사용하는 최적화 알고리즘에 해당

동작:
- 학습률을 지정하고 model.parameters()를 optimizer에 연결
"""
learning_rate = 1e-2
model = semiGPT(ko_vocab_size)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

In [21]:
"""
학습 루프 실행

목적:
- 미니배치 학습을 반복해 손실을 낮추도록 모델 파라미터를 업데이트

동작:
- 배치 생성 -> 순전파 -> gradient 초기화 -> 역전파 -> 파라미터 업데이트 순서로 진행
- tqdm으로 전체 진행률을 시각화하고 마지막 스텝의 loss를 출력
"""
from tqdm.auto import tqdm

batch_size = 32
for steps in tqdm(range(10000)):
    example_x, example_y = batch_function("train")
    logits, loss = model(example_x, example_y)
    # 옵티마이저 초기화 
    optimizer.zero_grad(set_to_none=True)
    # 역전파 계산 
    loss.backward()
    # 가중치 업데이트 
    optimizer.step()

print(loss.item())

100%|██████████| 10000/10000 [01:17<00:00, 129.17it/s]

3.436306953430176


In [22]:
"""
학습 후 텍스트 생성 확인

목적:
- 학습이 진행된 모델로 짧은 길이의 텍스트를 생성해 결과를 확인

동작:
- 시작 토큰(0)에서 generate를 호출하고, 생성된 id 시퀀스를 문자로 복원해 출력
"""
print(token_decode(model.generate(torch.zeros((1, 1), dtype=torch.long), max_new_tokens=10)[0].tolist()))

torch.Size([1, 2701])
torch.Size([1, 2701])
torch.Size([1, 2701])
torch.Size([1, 2701])
torch.Size([1, 2701])
torch.Size([1, 2701])
torch.Size([1, 2701])
torch.Size([1, 2701])
torch.Size([1, 2701])
torch.Size([1, 2701])
 콘텐츠첫鐵제시 3.


### 2.4.1 GPU로 데이터를 옮기는 방법

In [23]:
"""
실행 장치 선택

목적:
- CUDA 사용 가능 여부에 따라 연산 장치를 자동 선택

동작:
- GPU가 가능하면 cuda, 아니면 cpu 문자열을 device 변수에 저장
"""
device = "cuda" if torch.cuda.is_available() else "cpu"

In [24]:
"""
배치 데이터를 장치로 이동

목적:
- 배치 함수에서 생성한 입력/정답 텐서를 선택된 장치로 옮김

동작:
- x.to(device), y.to(device)를 적용해 모델 장치와 텐서 장치를 일치시킴
"""


def batch_function(mode):
    dataset = train_dataset if mode == "train" else test_dataset
    idx = torch.randint(len(dataset) - block_size, (batch_size,))
    x = torch.stack([dataset[index:index + block_size] for index in idx])
    y = torch.stack([dataset[index + 1:index + block_size + 1] for index in idx])
    x, y = x.to(device), y.to(device)  # .to 를 추가
    return x, y

### 2.4.2 Loss 함수 만들기

In [25]:
"""
평가용 손실 계산 함수 (장표)

목적:
- 학습 중 train/eval 평균 손실을 안정적으로 측정하는 유틸 함수를 정의

동작:
- no_grad 및 eval 모드에서 여러 배치를 평가하고 평균 손실을 계산
- 측정 후 다시 train 모드로 복귀
"""


@torch.no_grad()
def compute_loss_metrics():
    out = {}
    model.eval()
    for mode in ["train", "eval"]:
        losses = torch.zeros(eval_iteration)
        for k in range(eval_iteration):
            inputs, targets = batch_function(mode)
            logits, loss = model(inputs, targets)
            losses[k] = loss.item()
        out[mode] = losses.mean()
    model.train()
    return out

### 2.4.3 전체 코드 복습

In [49]:
"""
전체 학습 코드 통합 실행

목적:
- 지금까지의 설정, 모델, 학습/평가 루프를 하나로 합쳐 end-to-end로 실행

동작:
- 하이퍼파라미터 설정, 배치 함수/평가 함수/모델 정의를 한 셀에 정리
- 주기적으로 train/val loss를 출력하고 마지막에 텍스트 생성 결과를 확인
"""
import torch
import torch.nn as nn
import torch.nn.functional as F

batch_size = 32
block_size = 8
max_iteration = 50000
eval_interval = 300
learning_rate = 1e-2
device = "cuda" if torch.cuda.is_available() else "cpu"
eval_iteration = 200


# 학습/평가용 미니배치 (x, y)
# x는 길이 block_size 입력 시퀀스, y는 한 칸 오른쪽으로 민 정답 시퀀스
# 마지막에 device로 이동
def batch_function(mode):
    dataset = train_dataset if mode == "train" else test_dataset
    idx = torch.randint(len(dataset) - block_size, (batch_size,))
    x = torch.stack([dataset[index:index + block_size] for index in idx])
    y = torch.stack([dataset[index + 1:index + block_size + 1] for index in idx])
    x, y = x.to(device), y.to(device)  # .to 를 추가
    return x, y


# no_grad + eval() 상태에서 train/eval 각각 eval_iteration번 배치를 뽑아 평균 loss를 계산
# 끝나면 다시 train()으로 복귀
@torch.no_grad()
def compute_loss_metrics():
    out = {}
    model.eval()
    for mode in ["train", "eval"]:
        losses = torch.zeros(eval_iteration)
        for k in range(eval_iteration):
            inputs, targets = batch_function(mode)
            logits, loss = model(inputs, targets)
            losses[k] = loss.item()
        out[mode] = losses.mean()
    model.train()
    return out


# 토큰 ID를 바로 vocab 크기 점수로 바꾸는 임베딩 테이블(Embedding(vocab, vocab))을 정의
class semiGPT(nn.Module):
    def __init__(self, vocab_length):
        super().__init__()
        self.embedding_token_table = nn.Embedding(vocab_length, vocab_length)

    # targets가 없으면 logits만 반환(생성용), 있으면 logits/targets를 2D/1D로 펼쳐 cross_entropy loss까지 계산(학습용)
    def forward(self, inputs, targets=None):
        logits = self.embedding_token_table(inputs)
        if targets is None:
            loss = None
        else:
            batch, seq_length, vocab_length = logits.shape
            logits = logits.view(batch * seq_length, vocab_length)
            targets = targets.view(batch * seq_length)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    # 반복적으로 마지막 시점 logits을 softmax -> 샘플링(multinomial) -> 입력 뒤에 붙여 토큰을 확장
    def generate(self, inputs, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, loss = self.forward(inputs)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            next_inputs = torch.multinomial(probs, num_samples=1)
            inputs = torch.cat((inputs, next_inputs), dim=1)
        return inputs


model = semiGPT(ko_vocab_size).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for step in range(max_iteration):  # 0부터 max_iteration-1까지 학습 스텝 반복
    if step % eval_interval == 0:  # eval_interval 간격마다 평가 시점 판별
        losses = compute_loss_metrics()  # train/eval 평균 loss 측정해서 딕셔너리로 받음
        print(
            f'step : {step}, train loss : {losses["train"]:.4f}, val loss : {losses["eval"]:.4f}')  # 현재 스텝의 학습/검증 손실 출력

    example_x, example_y = batch_function("train")  # 학습 데이터에서 랜덤 미니배치 x,y 생성
    logits, loss = model(example_x, example_y)  # 순전파 수행, 예측 logits와 loss 계산
    optimizer.zero_grad(set_to_none=True)  # 이전 스텝 gradient 초기화, 메모리 효율 위해 None으로 설정
    loss.backward()  # 역전파 수행, 각 파라미터의 gradient 계산
    optimizer.step()  # 계산된 gradient 사용해서 파라미터 업데이트

inputs = torch.zeros((1, 1), dtype=torch.long, device=device)
print(token_decode(model.generate(inputs, max_new_tokens=100)[0].tolist()))

step : 0, train loss : 8.3307, val loss : 8.3297
step : 300, train loss : 6.0727, val loss : 6.0757
step : 600, train loss : 4.7857, val loss : 4.7613
step : 900, train loss : 4.2304, val loss : 4.2123
step : 1200, train loss : 3.9608, val loss : 3.9700
step : 1500, train loss : 3.8186, val loss : 3.7979
step : 1800, train loss : 3.7175, val loss : 3.7319
step : 2100, train loss : 3.6738, val loss : 3.6619
step : 2400, train loss : 3.6145, val loss : 3.6298
step : 2700, train loss : 3.5725, val loss : 3.5857
step : 3000, train loss : 3.5686, val loss : 3.5737
step : 3300, train loss : 3.5355, val loss : 3.5303
step : 3600, train loss : 3.5137, val loss : 3.5381
step : 3900, train loss : 3.4909, val loss : 3.5292
step : 4200, train loss : 3.5000, val loss : 3.4936
step : 4500, train loss : 3.4776, val loss : 3.4854
step : 4800, train loss : 3.4685, val loss : 3.4712
step : 5100, train loss : 3.4541, val loss : 3.4577
step : 5400, train loss : 3.4615, val loss : 3.4742
step : 5700, train

## 2.5 셀프 어텐션 추가하기

### 2.5.1	문자들 간의 정보를 주고받는 방식(평균 방식)

In [27]:
import torch

torch.manual_seed(1441)
num_batches, sequence_length, embedding_dim = 2, 4, 6
embeddings_tensor = torch.randn(num_batches,
                                sequence_length,
                                embedding_dim)
embeddings_tensor.shape

torch.Size([2, 4, 6])

In [28]:
# 이전 임베딩의 평균을 저장할 텐서 초기화
averaged_embeddings = torch.zeros((num_batches, sequence_length, embedding_dim))

# 각 배치에 대해 반복
for batch_index in range(num_batches):
    # 각 시퀀스 위치에 대해 반복
    for sequence_position in range(sequence_length):
        # 현재 시퀀스 위치까지의 이전 임베딩을 슬라이스
        previous_embeddings = embeddings_tensor[batch_index, :sequence_position + 1]
        # 현재 위치까지의 임베딩의 평균을 계산
        averaged_embeddings[batch_index, sequence_position] = torch.mean(
            previous_embeddings,
            dim=0
        )

# for 문을 사용하는 방식은 시간복잡도 문제로 인해 효율적이지 않음

In [29]:
print(embeddings_tensor[0])
print(averaged_embeddings[0])

tensor([[-1.1437, -1.2611, -0.1634, -0.5255, -1.0879,  0.3712],
        [ 2.2335,  0.3099, -1.3975,  1.1141, -0.3373,  0.6924],
        [ 0.2644,  1.1567, -0.5040, -0.7986,  2.6778,  1.4161],
        [ 1.3159, -0.5231,  1.2933, -0.8819,  0.7118,  0.4209]])
tensor([[-1.1437, -1.2611, -0.1634, -0.5255, -1.0879,  0.3712],
        [ 0.5449, -0.4756, -0.7804,  0.2943, -0.7126,  0.5318],
        [ 0.4514,  0.0685, -0.6883, -0.0700,  0.4175,  0.8266],
        [ 0.6675, -0.0794, -0.1929, -0.2730,  0.4911,  0.7252]])


### 2.5.2. 행렬곱 연산으로 더 빠르게 정보를 주고받기 

In [30]:
# 행렬곱 연산 예시

A = torch.ones(3, 3)
B = torch.randint(0, 10, (3, 2)).float()
AB = A @ B  # @ 연산자로 행렬 곱 수행

print(" A 행렬 ")
print(A)
print("==============")
print("==============")
print(" B 행렬 ")
print(B)
print("==============")
print("==============")
print(" AB 행렬 ")
print(AB)

 A 행렬 
tensor([[1., 1., 1.],
        [1., 1., 1.],
        [1., 1., 1.]])
 B 행렬 
tensor([[7., 2.],
        [0., 5.],
        [2., 2.]])
 AB 행렬 
tensor([[9., 9.],
        [9., 9.],
        [9., 9.]])


In [31]:
weight = torch.tril(torch.ones(sequence_length, sequence_length))
weight = weight.masked_fill(weight == 0, float('-inf'))  # 0이라는 숫자에는 -inf를 쓰우겠다는 코드이다.
print(weight)
weight = F.softmax(weight, dim=-1)
print(weight)

tensor([[1., -inf, -inf, -inf],
        [1., 1., -inf, -inf],
        [1., 1., 1., -inf],
        [1., 1., 1., 1.]])
tensor([[1.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500]])


### 2.5.3 셀프 어텐션이란?


In [32]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# 고정된 난수 시드 설정
torch.manual_seed(1111)

# 배치 크기, 시퀀스 길이, 채널 수 설정
batch_size, seq_length, num_channels = 2, 4, 4
input_tensor = torch.randn(batch_size, seq_length, num_channels)

# 각 헤드의 크기
head_size = 16

# Key, Query, Value 변환을 위한 선형 레이어
key_transform = nn.Linear(num_channels, head_size, bias=False)
query_transform = nn.Linear(num_channels, head_size, bias=False)
value_transform = nn.Linear(num_channels, head_size, bias=False)

# Key, Query, Value 변환 수행
keys = key_transform(input_tensor)
queries = query_transform(input_tensor)
values = value_transform(input_tensor)

# Attention 스코어 계산
attention_scores = queries @ keys.transpose(-2, -1)

# 하삼각행렬 생성 및 마스킹
mask_lower_triangle = torch.tril(torch.ones(seq_length, seq_length))
attention_scores = attention_scores.masked_fill(mask_lower_triangle == 0, float('-inf'))

# 소프트맥스 함수를 사용하여 확률 정규화
normalized_scores = F.softmax(attention_scores, dim=-1)

# 최종 출력 계산
output_tensor = normalized_scores @ values

output_tensor


tensor([[[-0.4755, -0.5409, -0.1864,  0.2951, -1.0717, -0.6172, -0.0176,
           0.1793, -0.1113,  0.6589, -0.4507, -0.1181, -0.9728, -0.8870,
           0.2349, -0.0431],
         [-0.4675, -0.5344, -0.1847,  0.2859, -1.0581, -0.6044, -0.0154,
           0.1778, -0.1141,  0.6524, -0.4473, -0.1211, -0.9561, -0.8733,
           0.2352, -0.0451],
         [-0.0760, -0.1545, -0.0268, -0.0634, -0.2490, -0.0492,  0.0418,
           0.0039, -0.1387,  0.1754, -0.1870, -0.1300, -0.1049, -0.1437,
           0.0797, -0.0811],
         [ 1.0050,  0.6488,  0.1280, -1.3952,  1.4225,  1.7320,  0.3957,
          -0.0998, -0.6179, -0.5368,  0.1755, -0.6712,  2.0809,  1.6208,
           0.2876, -0.4129]],

        [[-0.1629, -0.3577,  0.2200, -0.0743, -0.4798, -0.1531,  0.1460,
          -0.3159, -0.3507,  0.2564, -0.4777,  0.0395, -0.2861, -0.3503,
          -0.0974, -0.1463],
         [-0.1699, -0.3586,  0.1711, -0.0815, -0.4939, -0.1562,  0.1316,
          -0.2638, -0.3395,  0.2754, -0.4681, -0.0

### 2.5.4 왜 root d_k로 나누어야 하는가?


In [33]:
batch_size, sequence_length, embedding_dim = 2, 4, 4

k = torch.randn(batch_size, sequence_length, embedding_dim)
q = torch.randn(batch_size, sequence_length, embedding_dim)
wei = q @ k.transpose(-2, -1)
wei.var()

tensor(3.0996)

In [34]:
k = torch.randn(batch_size, sequence_length, embedding_dim)
q = torch.randn(batch_size, sequence_length, embedding_dim)

# 임베딩 차원의 제곱근으로 나눠 분산을 줄임
# 분산(Variance) : 데이터(변량)들이 평균을 중심으로 얼마나 흩어져 있는지를 나타내는 수치
wei = q @ k.transpose(-2, -1) * (embedding_dim ** -0.5)
wei.var()

tensor(0.7070)

In [35]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# 고정된 난수 시드 설정
torch.manual_seed(1111)

# 배치 크기, 시퀀스 길이, 채널 수 설정
batch_size, sequence_length, embedding_dim = 2, 4, 4
input_tensor = torch.randn(batch_size, sequence_length, embedding_dim)

# 헤드 사이즈 설정
head_dimension = 16

# Key, Query, Value 변환을 위한 선형 레이어
key_layer = nn.Linear(embedding_dim, head_dimension, bias=False)
query_layer = nn.Linear(embedding_dim, head_dimension, bias=False)
value_layer = nn.Linear(embedding_dim, head_dimension, bias=False)

# Key, Query, Value 변환 수행
key_matrix = key_layer(input_tensor)
query_matrix = query_layer(input_tensor)

# 스케일링 계수를 적용한 Attention 스코어 계산
scaling_factor = embedding_dim ** -0.5
attention_scores = query_matrix @ key_matrix.transpose(-2, -1) * scaling_factor

# 하삼각 행렬로 마스킹, 무한대로 채움
mask = torch.tril(torch.ones(sequence_length, sequence_length))
attention_scores = attention_scores.masked_fill(mask == 0, float('-inf'))

# 소프트맥스를 적용하여 Attention 확률 정규화
normalized_attention = F.softmax(attention_scores, dim=-1)

# Value 변환 적용
value_matrix = value_layer(input_tensor)

# 최종 출력 계산
output_tensor = normalized_attention @ value_matrix

output_tensor

tensor([[[-4.7553e-01, -5.4087e-01, -1.8645e-01,  2.9508e-01, -1.0717e+00,
          -6.1721e-01, -1.7619e-02,  1.7932e-01, -1.1134e-01,  6.5890e-01,
          -4.5073e-01, -1.1805e-01, -9.7278e-01, -8.8699e-01,  2.3494e-01,
          -4.3051e-02],
         [-3.7282e-01, -4.5845e-01, -1.6476e-01,  1.7766e-01, -8.9889e-01,
          -4.5412e-01,  1.1151e-02,  1.6013e-01, -1.4667e-01,  5.7623e-01,
          -4.0744e-01, -1.5664e-01, -7.6102e-01, -7.1314e-01,  2.3889e-01,
          -6.8812e-02],
         [ 3.3135e-02, -3.0254e-02,  3.8257e-02, -1.3334e-01,  1.8626e-02,
           8.7150e-02,  4.3045e-02, -7.2718e-02, -1.1493e-01, -2.8212e-03,
          -8.7858e-02, -9.4005e-02,  1.4480e-01,  7.8446e-02, -1.1284e-02,
          -7.3810e-02],
         [ 8.0965e-01,  5.1643e-01,  1.1648e-01, -1.1408e+00,  1.1586e+00,
           1.3968e+00,  3.1847e-01, -1.0840e-01, -5.1064e-01, -4.4907e-01,
           1.2734e-01, -5.5556e-01,  1.7125e+00,  1.3270e+00,  2.0701e-01,
          -3.4455e-01]],

  

### 2.5.5 셀프 어텐션 적용하기

In [36]:
"""
셀프 어텐션 모델 통합 학습 실행

목적:
- 단일 Head 셀프 어텐션을 포함한 문자 단위 언어모델을 끝까지 구성
- 학습 루프를 돌려 손실 하강과 생성 결과를 함께 확인

동작:
- Head 클래스에서 Q,K,V와 causal mask를 적용해 미래 토큰 참조를 차단
- semiGPT에서 token/position 임베딩을 합친 뒤 attention 출력으로 logits를 계산
- compute_loss_metrics로 train/eval 손실 평균을 주기적으로 측정
- optimizer로 반복 업데이트 후 generate로 샘플 텍스트를 생성
"""
import torch
import torch.nn as nn
import torch.nn.functional as F

batch_size = 32
block_size = 8
max_iteration = 50000
eval_interval = 300
learning_rate = 1e-2
device = "cuda" if torch.cuda.is_available() else "cpu"
eval_iteration = 200
n_embed = 32


class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embed, head_size, bias=False)
        self.query = nn.Linear(n_embed, head_size, bias=False)
        self.value = nn.Linear(n_embed, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))

    def forward(self, inputs):
        batch_size, sequence_length, embedding_dim = inputs.shape
        keys = self.key(inputs)
        queries = self.query(inputs)
        weights = queries @ keys.transpose(-2, -1) * (embedding_dim ** -0.5)
        weights = weights.masked_fill(
            self.tril[:sequence_length, :sequence_length] == 0, float("-inf")
        )
        weights = F.softmax(weights, dim=-1)
        values = self.value(inputs)
        output = weights @ values
        return output


def batch_function(mode):
    dataset = train_dataset if mode == "train" else test_dataset
    idx = torch.randint(len(dataset) - block_size, (batch_size,))
    x = torch.stack([dataset[index:index + block_size] for index in idx])
    y = torch.stack([dataset[index + 1:index + block_size + 1] for index in idx])
    x, y = x.to(device), y.to(device)
    return x, y


@torch.no_grad()
def compute_loss_metrics():
    out = {}
    model.eval()
    for mode in ["train", "eval"]:
        losses = torch.zeros(eval_iteration)
        for k in range(eval_iteration):
            inputs, targets = batch_function(mode)
            logits, loss = model(inputs, targets)
            losses[k] = loss.item()
        out[mode] = losses.mean()
    model.train()
    return out


class semiGPT(nn.Module):
    def __init__(self, vocab_length):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_length, n_embed)
        self.position_embedding_table = nn.Embedding(block_size, n_embed)
        self.attention_head = Head(n_embed)
        self.lm_head = nn.Linear(n_embed, vocab_length)

    def forward(self, inputs, targets=None):
        batch, sequence = inputs.shape

        token_embed = self.token_embedding_table(inputs)
        pos_embed = self.position_embedding_table(
            torch.arange(sequence, device=device)
        )
        x = token_embed + pos_embed
        x = self.attention_head(x)
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            batch, sequence, embed_size = logits.shape
            logits = logits.view(batch * sequence, embed_size)
            targets = targets.view(batch * sequence)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, inputs, max_new_tokens):
        for _ in range(max_new_tokens):
            inputs_cond = inputs[:, -block_size:]
            logits, loss = self(inputs_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            next_inputs = torch.multinomial(probs, num_samples=1)
            inputs = torch.cat((inputs, next_inputs), dim=1)
        return inputs


model = semiGPT(ko_vocab_size).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for step in range(max_iteration):
    if step % eval_interval == 0:
        losses = compute_loss_metrics()
        print(f'step : {step}, train loss : {losses["train"]:.4f}, val loss : {losses["eval"]:.4f}')

    example_x, example_y = batch_function("train")
    logits, loss = model(example_x, example_y)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

inputs = torch.zeros((1, 1), dtype=torch.long, device=device)
print("-----------------------------------------------")
print(token_decode(model.generate(inputs, max_new_tokens=100)[0].tolist()))


step : 0, train loss : 7.9190, val loss : 7.9190
step : 300, train loss : 4.1713, val loss : 4.1825
step : 600, train loss : 3.8909, val loss : 3.8957
step : 900, train loss : 3.7768, val loss : 3.7932
step : 1200, train loss : 3.7354, val loss : 3.7393
step : 1500, train loss : 3.6956, val loss : 3.6906
step : 1800, train loss : 3.6496, val loss : 3.6491
step : 2100, train loss : 3.6383, val loss : 3.6286
step : 2400, train loss : 3.6279, val loss : 3.6242
step : 2700, train loss : 3.6096, val loss : 3.5777
step : 3000, train loss : 3.5961, val loss : 3.5633
step : 3300, train loss : 3.5682, val loss : 3.5593
step : 3600, train loss : 3.5513, val loss : 3.5512
step : 3900, train loss : 3.5370, val loss : 3.5339
step : 4200, train loss : 3.5330, val loss : 3.5163
step : 4500, train loss : 3.5580, val loss : 3.5167
step : 4800, train loss : 3.5016, val loss : 3.5312
step : 5100, train loss : 3.5267, val loss : 3.5105
step : 5400, train loss : 3.5295, val loss : 3.5206
step : 5700, train

해당 결과 문장은 “정확한 의미를 전달하는 완성 문장”이라기보다,

**모델이 학습 데이터의 문자 패턴/주제어를 확률적으로 이어 붙인 샘플**

왜 문장 구성이 깨지느냐는 아래 이유가 큼

1. 문자 단위 토크나이징
- 모델이 단어/형태소가 아니라 문자 하나씩 예측해서, 말이 안 되는 조합이 쉽게 생기게 됨

2. 컨텍스트가 매우 짧음 (block_size=8)
- 바로 앞 8글자만 보고 다음 글자를 맞추므로, 긴 문장 문법/의미 일관성을 유지하기 어려움

3. 모델 용량이 작은 초기 구조
단일 attention head + 작은 임베딩(n_embed=32) 단계라 표현력이 제한적임

4. 샘플링 노이즈
생성 시 multinomial 샘플링을 써서 확률적으로 랜덤성이 들어가 문장이 더 흔들리게 됨

결론 : 위 출력 결과물은, “셀프 어텐션을 붙였더니 텍스트 패턴은 학습했지만, 아직 자연어 문장 품질은 낮은 단계” 라고 볼 수 있음

In [37]:
"""
셀프 어텐션 모델 추가 생성 샘플 확인

목적:
- 직전 학습 모델로 더 짧은 길이의 생성 결과를 추가 점검

동작:
- 시작 컨텍스트 inputs를 그대로 사용해 max_new_tokens=32로 생성
- token_decode 결과를 출력해 모델의 현재 문장 패턴을 확인
"""
print("-----------------------------------------------")
print(token_decode(model.generate(inputs, max_new_tokens=32)[0].tolist()))

-----------------------------------------------
 감하기보는 이 난달 기술 데 독앙회회의 였다. 302와 공


## 2.6 멀티헤드 어텐션과 피드포워드

### 2.6.1 멀티헤드 어텐션 만들기

In [38]:
"""
멀티헤드 어텐션 모델 통합 학습 실행

목적:
- 단일 Head를 MultiHeadAttention으로 확장한 모델 성능을 확인
- 학습률과 구조 변경이 손실 및 생성 결과에 주는 영향을 비교

동작:
- 여러 Head 출력을 concat한 뒤 projection으로 임베딩 차원으로 되돌림
- forward에서 logits/targets를 평탄화해 cross entropy 손실을 계산
- 주기적 평가와 파라미터 업데이트를 반복한 뒤 텍스트를 생성
"""
import torch
import torch.nn as nn
import torch.nn.functional as F

batch_size = 32
block_size = 8
max_iteration = 50000
eval_interval = 300
learning_rate = 1e-3
device = "cuda" if torch.cuda.is_available() else "cpu"
eval_iteration = 200
n_embed = 32


class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embed, head_size, bias=False)
        self.query = nn.Linear(n_embed, head_size, bias=False)
        self.value = nn.Linear(n_embed, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))

    def forward(self, inputs):
        batch_size, sequence_length, embedding_dim = inputs.shape
        keys = self.key(inputs)
        queries = self.query(inputs)
        weights = queries @ keys.transpose(-2, -1) * (embedding_dim ** -0.5)
        weights = weights.masked_fill(self.tril[:sequence_length, :sequence_length] == 0, float("-inf"))
        weights = F.softmax(weights, dim=-1)
        values = self.value(inputs)
        output = weights @ values
        return output


def batch_function(mode):
    dataset = train_dataset if mode == "train" else test_dataset
    idx = torch.randint(len(dataset) - block_size, (batch_size,))
    x = torch.stack([dataset[index:index + block_size] for index in idx])
    y = torch.stack([dataset[index + 1:index + block_size + 1] for index in idx])
    x, y = x.to(device), y.to(device)
    return x, y


@torch.no_grad()
def compute_loss_metrics():
    out = {}
    model.eval()
    for mode in ["train", "eval"]:
        losses = torch.zeros(eval_iteration)
        for k in range(eval_iteration):
            inputs, targets = batch_function(mode)
            logits, loss = model(inputs, targets)
            losses[k] = loss.item()
        out[mode] = losses.mean()
    model.train()
    return out


class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])

    def forward(self, inputs):
        return torch.cat([head(inputs) for head in self.heads], dim=-1)


class semiGPT(nn.Module):
    def __init__(self, vocab_length):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_length, n_embed)
        self.position_embedding_table = nn.Embedding(block_size, n_embed)

        # 멀티 헤드 어텐션 적용
        # 기존: self.attention_head = Head(n_embed)
        self.attention_head = MultiHeadAttention(4, n_embed // 4)
        self.lm_head = nn.Linear(n_embed, vocab_length)

    def forward(self, inputs, targets=None):
        batch, sequence = inputs.shape

        token_embed = self.token_embedding_table(inputs)
        pos_embed = self.position_embedding_table(torch.arange(sequence, device=device))
        x = token_embed + pos_embed
        x = self.attention_head(x)
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            batch, sequence, embed_size = logits.shape
            logits = logits.view(batch * sequence, embed_size)
            targets = targets.view(batch * sequence)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, inputs, max_new_tokens):
        for _ in range(max_new_tokens):
            inputs_cond = inputs[:, -block_size:]
            logits, loss = self(inputs_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            next_inputs = torch.multinomial(probs, num_samples=1)
            inputs = torch.cat((inputs, next_inputs), dim=1)
        return inputs


model = semiGPT(ko_vocab_size).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for step in range(max_iteration):
    if step % eval_interval == 0:
        losses = compute_loss_metrics()
        print(f'step : {step}, train loss : {losses["train"]:.4f}, val loss : {losses["eval"]:.4f}')

    example_x, example_y = batch_function("train")
    logits, loss = model(example_x, example_y)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

inputs = torch.zeros((1, 1), dtype=torch.long, device=device)
print("-----------------------------------------------")
print(token_decode(model.generate(inputs, max_new_tokens=100)[0].tolist()))

step : 0, train loss : 7.9309, val loss : 7.9298
step : 300, train loss : 4.7881, val loss : 4.7822
step : 600, train loss : 4.5738, val loss : 4.5813
step : 900, train loss : 4.4285, val loss : 4.4372
step : 1200, train loss : 4.3298, val loss : 4.3194
step : 1500, train loss : 4.2257, val loss : 4.2253
step : 1800, train loss : 4.1340, val loss : 4.1380
step : 2100, train loss : 4.0802, val loss : 4.0810
step : 2400, train loss : 4.0127, val loss : 4.0105
step : 2700, train loss : 3.9681, val loss : 3.9595
step : 3000, train loss : 3.9002, val loss : 3.8970
step : 3300, train loss : 3.8754, val loss : 3.8606
step : 3600, train loss : 3.8223, val loss : 3.8215
step : 3900, train loss : 3.7898, val loss : 3.7936
step : 4200, train loss : 3.7663, val loss : 3.7615
step : 4500, train loss : 3.7289, val loss : 3.7199
step : 4800, train loss : 3.7192, val loss : 3.6859
step : 5100, train loss : 3.7069, val loss : 3.6843
step : 5400, train loss : 3.6780, val loss : 3.6631
step : 5700, train

### 2.6.2 FeedForward

In [39]:
"""
FeedForward 추가 모델 통합 학습 실행

목적:
- 멀티헤드 어텐션 뒤에 위치별 MLP(FeedForward)를 추가해 표현력을 높임
- attention만 사용한 구조와 비교 가능한 학습 로그를 확보

동작:
- FeedForward를 n_embed -> 4*n_embed -> n_embed 두 층으로 구성합니
- attention 출력에 feedforward 출력을 연결해 시퀀스 특징을 재가공
- 동일한 학습/평가 루프로 손실 추이와 생성 문장을 확인
"""
import torch
import torch.nn as nn
import torch.nn.functional as F

batch_size = 32
block_size = 8
max_iteration = 50000
eval_interval = 300
learning_rate = 1e-2
device = "cuda" if torch.cuda.is_available() else "cpu"
eval_iteration = 200
n_embed = 32


class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embed, head_size, bias=False)
        self.query = nn.Linear(n_embed, head_size, bias=False)
        self.value = nn.Linear(n_embed, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))

    def forward(self, inputs):
        batch_size, sequence_length, embedding_dim = inputs.shape
        keys = self.key(inputs)
        queries = self.query(inputs)
        weights = queries @ keys.transpose(-2, -1) * (embedding_dim ** -0.5)
        weights = weights.masked_fill(self.tril[:sequence_length, :sequence_length] == 0, float("-inf"))
        weights = F.softmax(weights, dim=-1)
        values = self.value(inputs)
        output = weights @ values
        return output


def batch_function(mode):
    dataset = train_dataset if mode == "train" else test_dataset
    idx = torch.randint(len(dataset) - block_size, (batch_size,))
    x = torch.stack([dataset[index:index + block_size] for index in idx])
    y = torch.stack([dataset[index + 1:index + block_size + 1] for index in idx])
    x, y = x.to(device), y.to(device)
    return x, y


@torch.no_grad()
def compute_loss_metrics():
    out = {}
    model.eval()
    for mode in ["train", "eval"]:
        losses = torch.zeros(eval_iteration)
        for k in range(eval_iteration):
            inputs, targets = batch_function(mode)
            logits, loss = model(inputs, targets)
            losses[k] = loss.item()
        out[mode] = losses.mean()
    model.train()
    return out


class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])

    def forward(self, inputs):
        return torch.cat([head(inputs) for head in self.heads], dim=-1)


class FeedForward(nn.Module):
    def __init__(self, n_embed):
        super().__init__()
        self.layer = nn.Sequential(
            nn.Linear(n_embed, 4 * n_embed),
            nn.ReLU(),
            nn.Linear(4 * n_embed, n_embed)
        )

    def forward(self, input_tensor):
        return self.layer(input_tensor)


class semiGPT(nn.Module):
    def __init__(self, vocab_length):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_length, n_embed)
        self.position_embedding_table = nn.Embedding(block_size, n_embed)
        self.attention_head = MultiHeadAttention(4, n_embed // 4)

        # 멀티헤드 어텐션 뒤에 위치별 MLP(FeedForward)를 추가해 표현력을 높임
        self.feed_forward = FeedForward(n_embed)
        self.lm_head = nn.Linear(n_embed, vocab_length)

    def forward(self, inputs, targets=None):
        batch, sequence = inputs.shape

        token_embed = self.token_embedding_table(inputs)
        pos_embed = self.position_embedding_table(torch.arange(sequence, device=device))
        x = token_embed + pos_embed
        x = self.attention_head(x)
        x = self.feed_forward(x)
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            batch, sequence, embed_size = logits.shape
            logits = logits.view(batch * sequence, embed_size)
            targets = targets.view(batch * sequence)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, inputs, max_new_tokens):
        for _ in range(max_new_tokens):
            inputs_cond = inputs[:, -block_size:]

            logits, loss = self(inputs_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            next_inputs = torch.multinomial(probs, num_samples=1)
            inputs = torch.cat((inputs, next_inputs), dim=1)
        return inputs


model = semiGPT(ko_vocab_size).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for step in range(max_iteration):
    if step % eval_interval == 0:
        losses = compute_loss_metrics()
        print(f'step : {step}, train loss : {losses["train"]:.4f}, val loss : {losses["eval"]:.4f}')

    example_x, example_y = batch_function("train")
    logits, loss = model(example_x, example_y)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

inputs = torch.zeros((1, 1), dtype=torch.long, device=device)
print("-----------------------------------------------")
print(token_decode(model.generate(inputs, max_new_tokens=100)[0].tolist()))

step : 0, train loss : 7.9046, val loss : 7.9043
step : 300, train loss : 4.2300, val loss : 4.2470
step : 600, train loss : 3.9763, val loss : 3.9662
step : 900, train loss : 3.8419, val loss : 3.8583
step : 1200, train loss : 3.7425, val loss : 3.7356
step : 1500, train loss : 3.6953, val loss : 3.7085
step : 1800, train loss : 3.6426, val loss : 3.6727
step : 2100, train loss : 3.6469, val loss : 3.6341
step : 2400, train loss : 3.6067, val loss : 3.5866
step : 2700, train loss : 3.5852, val loss : 3.5634
step : 3000, train loss : 3.5553, val loss : 3.5539
step : 3300, train loss : 3.5215, val loss : 3.5231
step : 3600, train loss : 3.5148, val loss : 3.5244
step : 3900, train loss : 3.5254, val loss : 3.5284
step : 4200, train loss : 3.5139, val loss : 3.5180
step : 4500, train loss : 3.5112, val loss : 3.4972
step : 4800, train loss : 3.5017, val loss : 3.4900
step : 5100, train loss : 3.4815, val loss : 3.4769
step : 5400, train loss : 3.4774, val loss : 3.4849
step : 5700, train

## 2.7 Blocks 만들기

In [40]:
"""
Transformer Block 스택 모델 통합 학습 실행

목적:
- MultiHeadAttention + FeedForward + Residual + LayerNorm 블록을 깊게 쌓음
- n_layer, n_head, dropout을 포함한 본격 구조로 학습 안정성을 점검

동작:
- Block 클래스에서 pre-norm과 residual 연결을 사용해 정보 흐름을 보존
- nn.Sequential로 여러 Block을 적층해 표현 깊이를 확장
- 최종 LayerNorm과 lm_head로 logits를 만들고 손실을 계산
- 반복 학습 후 생성 결과를 출력해 품질 변화를 확인
"""
import torch
import torch.nn as nn
import torch.nn.functional as F

batch_size = 32
block_size = 8
max_iteration = 50000
eval_interval = 300
learning_rate = 1e-2
device = "cuda" if torch.cuda.is_available() else "cpu"
eval_iteration = 200
n_embed = 32
n_head = 4
n_layer = 4
dropout = 0.1


class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embed, head_size, bias=False)
        self.query = nn.Linear(n_embed, head_size, bias=False)
        self.value = nn.Linear(n_embed, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))

    def forward(self, inputs):
        batch_size, sequence_length, embedding_dim = inputs.shape
        keys = self.key(inputs)
        queries = self.query(inputs)
        weights = queries @ keys.transpose(-2, -1) * (embedding_dim ** -0.5)
        weights = weights.masked_fill(self.tril[:sequence_length, :sequence_length] == 0, float("-inf"))
        weights = F.softmax(weights, dim=-1)
        values = self.value(inputs)
        output = weights @ values
        return output


def batch_function(mode):
    dataset = train_dataset if mode == "train" else test_dataset
    idx = torch.randint(len(dataset) - block_size, (batch_size,))
    x = torch.stack([dataset[index:index + block_size] for index in idx])
    y = torch.stack([dataset[index + 1:index + block_size + 1] for index in idx])
    x, y = x.to(device), y.to(device)
    return x, y


@torch.no_grad()
def compute_loss_metrics():
    out = {}
    model.eval()
    for mode in ["train", "eval"]:
        losses = torch.zeros(eval_iteration)
        for k in range(eval_iteration):
            inputs, targets = batch_function(mode)
            logits, loss = model(inputs, targets)
            losses[k] = loss.item()
        out[mode] = losses.mean()
    model.train()
    return out


class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])

    def forward(self, inputs):
        return torch.cat([head(inputs) for head in self.heads], dim=-1)


class FeedForward(nn.Module):
    def __init__(self, n_embed):
        super().__init__()
        self.layer = nn.Sequential(
            nn.Linear(n_embed, 4 * n_embed),
            nn.ReLU(),
            nn.Linear(4 * n_embed, n_embed),
            nn.Dropout(dropout),
        )

    def forward(self, input_tensor):
        return self.layer(input_tensor)


class Block(nn.Module):  # Transformer의 기본 블록(Attention + FFN + Residual + LayerNorm)
    def __init__(self, n_embed, n_heads):  # n_embed: 임베딩 차원(C), n_heads: 어텐션 헤드 수
        super().__init__()  # nn.Module 초기화(파라미터/서브모듈 등록 기반)
        head_size = n_embed // n_heads  # 헤드 1개가 담당할 차원; 보통 n_embed가 n_heads로 나누어떨어져야 함
        self.attention = MultiHeadAttention(n_heads, head_size)
        # 멀티헤드 셀프어텐션 모듈
        # 입력 shape (B, T, C=n_embed) -> 출력 shape (B, T, C) 를 기대

        self.feed_forward = FeedForward(n_embed)
        # 위치별(토큰별) MLP/FFN
        # 각 토큰 벡터를 독립적으로 비선형 변환해 표현력 강화

        self.layer_norm1 = nn.LayerNorm(n_embed)
        # Attention 전에 적용하는 LayerNorm (Pre-Norm 구조)

        self.layer_norm2 = nn.LayerNorm(n_embed)
        # FeedForward 전에 적용하는 LayerNorm (Pre-Norm 구조)

    def forward(self, input_tensor):  # input_tensor shape: (B, T, C)
        input_tensor = input_tensor + self.attention(self.layer_norm1(input_tensor))
        # 1) layer_norm1으로 정규화
        # 2) self-attention 계산
        # 3) 원래 입력과 더하는 residual(skip) 연결
        # -> 학습 안정화, 깊은 네트워크에서 gradient 흐름 개선

        input_tensor = input_tensor + self.feed_forward(self.layer_norm2(input_tensor))
        # 1) layer_norm2로 정규화
        # 2) FFN 변환
        # 3) residual 연결
        # -> attention이 만든 표현을 비선형적으로 정제

        return input_tensor  # 최종 출력 shape 유지: (B, T, C)


class semiGPT(nn.Module):
    def __init__(self, vocab_length):
        super().__init__()
        self.embedding_token_table = nn.Embedding(vocab_length, n_embed)
        self.position_embedding_table = nn.Embedding(block_size, n_embed)

        # Block 생성
        self.blocks = nn.Sequential(*[Block(n_embed, 4) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embed)
        self.lm_head = nn.Linear(n_embed, vocab_length)

    def forward(self, inputs, targets=None):
        batch, sequence = inputs.shape

        token_embed = self.embedding_token_table(inputs)  # (B, T, C)
        pos_embed = self.position_embedding_table(torch.arange(sequence, device=device))  # (T, C)
        x = token_embed + pos_embed
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            batch, sequence, embed_size = logits.shape
            logits = logits.view(batch * sequence, embed_size)
            targets = targets.view(batch * sequence)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, inputs, max_new_tokens):
        for _ in range(max_new_tokens):
            inputs_cond = inputs[:, -block_size:]

            logits, loss = self(inputs_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            next_inputs = torch.multinomial(probs, num_samples=1)
            inputs = torch.cat((inputs, next_inputs), dim=1)
        return inputs


model = semiGPT(ko_vocab_size).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for step in range(max_iteration):
    if step % eval_interval == 0:
        losses = compute_loss_metrics()
        print(f'step : {step}, train loss : {losses["train"]:.4f}, val loss : {losses["eval"]:.4f}')

    example_x, example_y = batch_function("train")
    logits, loss = model(example_x, example_y)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

inputs = torch.zeros((1, 1), dtype=torch.long, device=device)
print("-----------------------------------------------")
print(token_decode(model.generate(inputs, max_new_tokens=100)[0].tolist()))

step : 0, train loss : 8.0116, val loss : 8.0067
step : 300, train loss : 4.1026, val loss : 4.0918
step : 600, train loss : 3.8108, val loss : 3.8128
step : 900, train loss : 3.6781, val loss : 3.6673
step : 1200, train loss : 3.6040, val loss : 3.6246
step : 1500, train loss : 3.5632, val loss : 3.5641
step : 1800, train loss : 3.5034, val loss : 3.5172
step : 2100, train loss : 3.4774, val loss : 3.4721
step : 2400, train loss : 3.4620, val loss : 3.4474
step : 2700, train loss : 3.4548, val loss : 3.4356
step : 3000, train loss : 3.4161, val loss : 3.4354
step : 3300, train loss : 3.3885, val loss : 3.4140
step : 3600, train loss : 3.3858, val loss : 3.3835
step : 3900, train loss : 3.3808, val loss : 3.3952
step : 4200, train loss : 3.3743, val loss : 3.3590
step : 4500, train loss : 3.3747, val loss : 3.3690
step : 4800, train loss : 3.3607, val loss : 3.3426
step : 5100, train loss : 3.3512, val loss : 3.3534
step : 5400, train loss : 3.3440, val loss : 3.3331
step : 5700, train

In [41]:
"""
사용자 시작어 기반 텍스트 생성

목적:
- 임의 시작 토큰 대신 실제 단어 프롬프트로 생성 동작을 확인

동작:
- input_word를 문자 id 시퀀스로 변환해 텐서를 만듬
- model.generate로 이어질 토큰을 확장
- ids_to_character로 디코딩해 사람이 읽을 문장으로 복원
"""
input_word = "의사"
input_ids = [character_to_ids[char] for char in input_word if char in character_to_ids]

# 입력 텐서 생성
inputs = torch.tensor([input_ids], dtype=torch.long).to(device)

# 모델을 사용하여 텍스트 생성
outputs = model.generate(inputs, 100)

# 생성된 결과 디코딩
generated_text = "".join([ids_to_character.get(i, '') for i in outputs[0].tolist()])

print("-----------------------------------------------")
print("Generated Text: ", generated_text)

-----------------------------------------------
Generated Text:  의사 입병폐전 이전경쟁력 모드핏 보전 화의 변들나 가능증가자 민원 앞으로 빠질 것으로 해외를 포함한 기대 낮아진 문합 이용자 2033년 최상 이주협만포준 미스오는 전환사에서 부콜 대통


In [42]:
dataset

DatasetDict({
    train: Dataset({
        features: ['date', 'category', 'press', 'title', 'document', 'link', 'summary'],
        num_rows: 22194
    })
    validation: Dataset({
        features: ['date', 'category', 'press', 'title', 'document', 'link', 'summary'],
        num_rows: 2466
    })
    test: Dataset({
        features: ['date', 'category', 'press', 'title', 'document', 'link', 'summary'],
        num_rows: 2740
    })
})

In [43]:
"""


목적:
- 토크나이저 학습 입력으로 사용할 원문 텍스트 목록을 준비

동작:
- train split 각 샘플에서 document 필드만 추출해 texts 리스트를 만듬
"""
texts = [example['document'] for example in dataset['train']]

In [44]:
"""
추출 텍스트 샘플 미리보기

목적:
- 토크나이저 학습에 들어갈 텍스트 형식을 사전 확인

동작:
- texts의 앞부분 두 항목을 출력해 전처리 상태를 점검
"""
texts[0:2]

['앵커 정부가 올해 하반기 우리 경제의 버팀목인 수출 확대를 위해 총력을 기울이기로 했습니다. 특히 수출 중소기업의 물류난 해소를 위해 무역금융 규모를 40조 원 이상 확대하고 물류비 지원과 임시선박 투입 등을 추진하기로 했습니다. 류환홍 기자가 보도합니다. 기자 수출은 최고의 실적을 보였지만 수입액이 급증하면서 올해 상반기 우리나라 무역수지는 역대 최악인 103억 달러 적자를 기록했습니다. 정부가 수출확대에 총력을 기울이기로 한 것은 원자재 가격 상승 등 대외 리스크가 가중되는 상황에서 수출 증가세 지속이야말로 한국경제의 회복을 위한 열쇠라고 본 것입니다. 추경호 경제부총리 겸 기획재정부 장관 정부는 우리 경제의 성장엔진인 수출이 높은 증가세를 지속할 수 있도록 총력을 다하겠습니다. 우선 물류 부담 증가 원자재 가격 상승 등 가중되고 있는 대외 리스크에 대해 적극 대응하겠습니다. 특히 중소기업과 중견기업 수출 지원을 위해 무역금융 규모를 연초 목표보다 40조 원 늘린 301조 원까지 확대하고 물류비 부담을 줄이기 위한 대책도 마련했습니다. 이창양 산업통상자원부 장관 국제 해상운임이 안정될 때까지 월 4척 이상의 임시선박을 지속 투입하는 한편 중소기업 전용 선복 적재 용량 도 현재보다 주당 50TEU 늘려 공급하겠습니다. 하반기에 우리 기업들의 수출 기회를 늘리기 위해 2 500여 개 수출기업을 대상으로 해외 전시회 참가를 지원하는 등 마케팅 지원도 벌이기로 했습니다. 정부는 또 이달 중으로 반도체를 비롯한 첨단 산업 육성 전략을 마련해 수출 증가세를 뒷받침하고 에너지 소비를 줄이기 위한 효율화 방안을 마련해 무역수지 개선에 나서기로 했습니다. YTN 류환홍입니다.',
 '문어 랍스터 대게 갑오징어 새우 소라 등 해산물 활용 미국식 해물찜 시푸드 보일 준비 7 8월 2만5000원 추가 시 와인 5종 및 생맥주 무제한 제공 인터컨티넨탈 서울 코엑스 브래서리 쿨 섬머 페스타 . 인터컨티넨탈 서울 코엑스 1층 뷔페 레스토랑 브래서리는 오는 6일부터 8월31일까지 

# 2.8 토크나이저 만들기

In [51]:
"""
BPE 토크나이저 학습 및 저장 (Byte Pair Encoding 토크나이저 모델)

목적:
- Hugging Face tokenizers 기반으로 한국어 뉴스용 BPE 토크나이저를 학습
- 학습된 tokenizer를 fast tokenizer 형식으로 저장하고 동작을 검증

동작:
- BPE 모델, trainer, special token, vocab size를 설정
- batch_iterator로 train 문서를 배치 단위 공급해 tokenizer를 학습
- vocab/merges와 tokenizer_config 파일을 저장해 재사용 가능 상태로 만듬
- 예시 문장을 encode/decode해 id와 토큰 분할 결과를 확인
"""
import os
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
from datasets import load_dataset
from transformers import PreTrainedTokenizerFast

# 저장할 디렉토리 경로 설정
SAVE_DIR = "./artifacts/tokenizer"

# 디렉토리가 없으면 생성
os.makedirs(SAVE_DIR, exist_ok=True)

# 원하는 어휘 크기 설정
VOCAB_SIZE = 30000

# 토크나이저 초기화
tokenizer = Tokenizer(BPE(unk_token="<unk>"))
tokenizer.pre_tokenizer = Whitespace()

# 트레이너 준비 (vocab_size 지정)
trainer = BpeTrainer(
    special_tokens=["<unk>", "<s>", "</s>", "<pad>"],
    vocab_size=VOCAB_SIZE
)


# 토크나이저 학습
def batch_iterator(batch_size=1000):
    for i in range(0, len(dataset["train"]), batch_size):
        yield dataset["train"][i: i + batch_size]["document"]


tokenizer.train_from_iterator(batch_iterator(), trainer=trainer)

# 토크나이저를 JSON 파일로 저장
tokenizer_path = os.path.join(SAVE_DIR, "tokenizer.json")
tokenizer.save(tokenizer_path)

# 토크나이저를 Hugging Face 형식으로 변환
huggingface_tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=tokenizer,
    unk_token="<unk>",
    bos_token="<s>",
    eos_token="</s>",
    pad_token="<pad>"
)

# Hugging Face 형식의 토크나이저 저장
huggingface_path = os.path.join(SAVE_DIR, "huggingface_tokenizer")
huggingface_tokenizer.save_pretrained(huggingface_path)

# Hugging Face 형식의 토크나이저 로드
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(huggingface_path)

# 어휘 크기 확인
print(f"Vocabulary size: {len(tokenizer.get_vocab())}")

# 테스트
test_texts = ["안녕하세요", "자연어 처리는 매우 흥미로운 분야입니다", "인공지능과 기계학습의 발전이 놀랍습니다"]
for text in test_texts:
    encoded = tokenizer.encode(text)
    print(f"Original: {text}")
    print(f"Encoded: {encoded}")
    print(f"Decoded: {tokenizer.decode(encoded)}")
    print(f"Tokens: {tokenizer.convert_ids_to_tokens(encoded)}")
    print()




Vocabulary size: 30000
Original: 안녕하세요
Encoded: [29138]
Decoded: 안녕하세요
Tokens: ['안녕하세요']

Original: 자연어 처리는 매우 흥미로운 분야입니다
Encoded: [22456, 2242, 2982, 4637, 16319, 3063, 2931, 2949]
Decoded: 자연어 처 리는 매우 흥미 로운 분야 입니다
Tokens: ['자연어', '처', '리는', '매우', '흥미', '로운', '분야', '입니다']

Original: 인공지능과 기계학습의 발전이 놀랍습니다
Encoded: [3765, 982, 5093, 5017, 2063, 22177, 1177, 1394, 2727]
Decoded: 인공지능 과 기계 학습 의 발전이 놀 랍 습니다
Tokens: ['인공지능', '과', '기계', '학습', '의', '발전이', '놀', '랍', '습니다']



In [47]:
"""
토크나이저 기반 Transformer 학습 파이프라인 실행

목적:
- 문자 사전 대신 BPE 토큰 id를 사용해 학습 데이터셋과 모델을 재구성
- DataLoader 기반 미니배치 학습과 프롬프트 생성까지 전체 흐름을 검증

동작:
- 토큰화 결과를 고정 길이 블록으로 자른 뒤 TensorDataset/DataLoader를 만듬
- Head, MultiHeadAttention, FeedForward, Block, GPTLanguageModel 클래스를 정의
- 학습 루프에서 손실을 출력하며 optimizer step을 반복
- 학습 후 입력 프롬프트를 encode해 generate하고 decode 결과를 확인
"""
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

# 하이퍼파라미터
batch_size = 32
block_size = 8
max_iteration = 100
eval_interval = 10
learning_rate = 1e-2
device = "cuda" if torch.cuda.is_available() else "cpu"
eval_iteration = 10
n_embed = 32
n_head = 4
n_layer = 4
dropout = 0.1


class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embed, head_size, bias=False)
        self.query = nn.Linear(n_embed, head_size, bias=False)
        self.value = nn.Linear(n_embed, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))

    def forward(self, inputs):
        batch_size, sequence_length, embedding_dim = inputs.shape
        keys = self.key(inputs)
        queries = self.query(inputs)
        weights = queries @ keys.transpose(-2, -1) * (embedding_dim ** -0.5)
        weights = weights.masked_fill(self.tril[:sequence_length, :sequence_length] == 0, float("-inf"))
        weights = F.softmax(weights, dim=-1)
        values = self.value(inputs)
        output = weights @ values
        return output


class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])

    def forward(self, inputs):
        return torch.cat([head(inputs) for head in self.heads], dim=-1)


class FeedForward(nn.Module):
    def __init__(self, n_embed):
        super().__init__()
        self.layer = nn.Sequential(
            nn.Linear(n_embed, 4 * n_embed),
            nn.ReLU(),
            nn.Linear(4 * n_embed, n_embed),
            nn.Dropout(dropout),
        )

    def forward(self, input_tensor):
        return self.layer(input_tensor)


class Block(nn.Module):
    def __init__(self, n_embed, n_heads):
        super().__init__()
        head_size = n_embed // n_heads
        self.attention = MultiHeadAttention(n_heads, head_size)
        self.feed_forward = FeedForward(n_embed)
        self.layer_norm1 = nn.LayerNorm(n_embed)
        self.layer_norm2 = nn.LayerNorm(n_embed)

    def forward(self, input_tensor):
        input_tensor = input_tensor + self.attention(self.layer_norm1(input_tensor))
        input_tensor = input_tensor + self.feed_forward(self.layer_norm2(input_tensor))
        return input_tensor


# 데이터셋 전처리
def preprocess_dataset(dataset, tokenizer):
    encoded_data = [tokenizer.encode(text, add_special_tokens=False) for text in dataset]
    tensor_data = [torch.tensor(seq, dtype=torch.long) for seq in encoded_data if len(seq) >= block_size + 1]
    return tensor_data


def create_dataloader(tensor_data, batch_size, block_size):
    dataset = TensorDataset(
        torch.stack([seq[:block_size] for seq in tensor_data]).to(device),
        torch.stack([seq[1:block_size + 1] for seq in tensor_data]).to(device)
    )
    return DataLoader(dataset, batch_size=batch_size, shuffle=True)


class semiGPT(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, n_embed)
        self.position_embedding = nn.Embedding(block_size, n_embed)
        self.blocks = nn.ModuleList([Block(n_embed, n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embed)
        self.lm_head = nn.Linear(n_embed, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        token_emb = self.token_embedding(idx)
        pos_emb = self.position_embedding(torch.arange(T, device=device))
        x = token_emb + pos_emb
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(B * T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx


# 데이터 전처리
# 토크나이저를 사용하여 처리 (기존에는 batch function을 사용)
n = int(0.9 * len(dataset["train"]["document"]))
train_data = preprocess_dataset(dataset["train"]["document"][:n], tokenizer)
test_data = preprocess_dataset(dataset["train"]["document"][n:], tokenizer)

# 데이터 로더 생성
train_loader = create_dataloader(train_data, batch_size, block_size)
test_loader = create_dataloader(test_data, batch_size, block_size)

# 모델 초기화
vocab_size = len(tokenizer.get_vocab())
model = semiGPT(vocab_size).to(device)
print(f"모델의 파라미터 수: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M")

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)


# 평가 함수
@torch.no_grad()
def evaluate(data_loader):
    model.eval()
    total_loss = 0
    for batch in data_loader:
        x, y = batch
        x, y = x.to(device), y.to(device)
        logits, loss = model(x, y)
        total_loss += loss.item()
    return total_loss / len(data_loader)


# 학습 루프
from tqdm.auto import tqdm

for step in tqdm(range(max_iteration)):
    if step % eval_interval == 0:
        train_loss = evaluate(train_loader)
        val_loss = evaluate(test_loader)
        print(f'step : {step}, train loss : {train_loss:.4f}, val loss : {val_loss:.4f}')

    model.train()
    for batch in train_loader:
        x, y = batch
        x, y = x.to(device), y.to(device)
        logits, loss = model(x, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

# 텍스트 생성
context = "의사"
context_encoded = tokenizer.encode(context, return_tensors='pt').to(device)
generated_ids = model.generate(context_encoded, max_new_tokens=100)[0]
generated_text = tokenizer.decode(generated_ids)
print("Generated Text:", generated_text)

모델의 파라미터 수: 0.70M


  0%|          | 0/100 [00:00<?, ?it/s]

step : 0, train loss : 9.3709, val loss : 9.3743


 10%|█         | 10/100 [00:58<08:29,  5.66s/it]

step : 10, train loss : 3.7252, val loss : 5.7810


...

step : 80, train loss : 3.0730, val loss : 6.4196


 90%|█████████ | 90/100 [08:52<00:56,  5.66s/it]

step : 90, train loss : 3.0669, val loss : 6.4667


100%|██████████| 100/100 [09:50<00:00,  5.91s/it]

Generated Text: 의사 정비사업 에 첫 회 콘텐츠 회 2022년 세계 이른 폭염 부동산 매수세가 기존 속 을 맞아 할 식을 전달 해 고 승 환 거쳐 11월 들이 측 최고치 를 었으나 넘어서 지지 소형 이 화상 심 규 에이 기자 오른쪽 이 발사 보일 것으로 전망된다 처음으로 1만 30 1조 700 전국 아파트 매매 거래가 전망 가격은 10 10 3억달러 2000 9 % 증가 9 2021년 메리츠 형 주 가 엘리 와 러 스 인더 스트리 포트 럼 제공 티브 해양 연세대 코리아 스마 전 25 60 에 있는 10일 연세대 대해 2분기 운영 기계 뱅 에 찾아 는 글로벌 비즈니스
